# 06 Linux 系统基础与常用命令

> 覆盖范围：Linux 体系结构（内核/系统调用/用户态）、文件系统层次（FHS）、文件与权限（chmod/umask/ACL）、常用命令（文件/进程/网络/文本）、文本处理三剑客（grep/sed/awk）、进程管理（信号/daemon）、网络排查（端口/连接）、shell 脚本（变量/管道/通配符）、系统监控（top/free/df）。
> 使用方式：先读**原理讲解**（内核如何隔离、权限数字怎么算、管道怎么跑），再看**命令示例**（默写），最后用 Python 模拟核心语义验证，自测面试追问。

---

## 1. Linux 体系结构：三层视图

```
┌─────────────────────────────┐
│  用户态：应用 / shell / 库      │   （你的程序、bash、python）
├─────────────────────────────┤
│  系统调用接口 syscall          │   （read/write/fork/exec…）
├─────────────────────────────┤
│  内核态：进程/内存/文件/网络/驱动 │   （权限最高，直接管理硬件）
└─────────────────────────────┘
```

- **用户态 → 内核态**：应用不能直接碰硬件，必须通过**系统调用（syscall）**（`read`/`write`/`fork`/`exec`/`mmap`/`socket`…）；
- **为什么分层**：安全隔离——普通进程无法越权访问他人内存或硬件；内核是唯一可信层；
- **中断与异常**：硬件中断（如网卡收包）与软中断让内核接管控制流。

## 2. 文件系统层次标准（FHS）

**一切皆文件**：设备、管道、socket 都以文件形式出现在 `/dev`、`/proc` 等特殊目录中。

| 路径 | 内容 | 面试点 |
| --- | --- | --- |
| `/bin` `/sbin` | 基本命令（系统启动就需要） | 现在多是 → /usr/bin 的软链 |
| `/usr` | 用户安装的程序/库/头文件 | `/usr/local` 常放自编译软件 |
| `/etc` | **配置文件**（不在 /etc 的程序很少） | 改配置后通常要重启服务 |
| `/var` | 动态数据（日志/缓存/队列） | `/var/log` 日志、`/var/tmp` |
| `/tmp` | 临时文件 | 重启清空；权限 1777（sticky） |
| `/home` | 普通用户家目录 | `/root` 是 root 的家 |
| `/proc` | **运行时内核信息**（虚拟文件系统） | `cat /proc/meminfo`、`/proc/cpuinfo`；每进程一个目录 |
| `/sys` | 内核设备模型 | 与 /proc 类似但更结构化 |
| `/dev` | 设备文件 | `/dev/null`、`/dev/zero` |

## 3. 文件与权限（高频）

### 3.1 权限位与数字表示

```
-rwxr-xr--  3  user  group  4096  Jan 1 12:00  file.sh
└┬┘└──┬──┘└─┬┘ └──┬──┘            └──────────── 文件名
类型 rwx 属主 属组                     （owner: rwx, group: r-x, other: r--）
```

数字权限 = 每位的和：**r=4, w=2, x=1**。`chmod 755` = 属主 rwx(7) + 属组 r-x(5) + 其他 r-x(5)。

```bash
chmod 755 script.sh        # 数字法
chmod u+x script.sh        # 符号法 u=owner g=group o=other a=all；+-= 增减赋
chown user:group file      # 改属主/属组
umask 022                  # 默认权限掩码：文件 666-022=644，目录 777-022=755
```

### 3.2 特殊权限

- **SUID（4）**：`rws`——执行时以**文件属主**身份运行（如 `/usr/bin/passwd`）；
- **SGID（2）**：继承属组；目录上设 SGID 时新建文件继承目录属组；
- **Sticky（1）**：`/tmp` 的 `t`——只有属主/root 能删除自己的文件。

## 4. 高频命令分类速记

```bash
# 文件操作
ls -lah / cp -r src dst / mv / rm -rf / touch / find / -name "*.py"
cat / less / head -n 5 / tail -f log   # tail -f 跟日志
ln -s target link                      # 软链接（符号链接，可跨文件系统）
du -sh / df -h                         # 目录大小 / 磁盘占用

# 进程管理
ps -ef | grep python                   # 查进程（全格式 + 过滤）
ps aux --sort=-%mem                     # 按内存排序
top / htop                              # 动态监控（q 退出）
kill -9 PID                            # 强杀（-9 SIGKILL 不可捕获）；kill PID 默认 SIGTERM
nohup cmd &                             # 后台运行，退出终端不挂
jobs / fg / bg                          # 任务控制

# 网络
ip addr / ifconfig                     # 网卡 IP
ss -tlnp / netstat -tlnp               # 监听端口 + 进程
curl -v http://host:port               # 测试接口（-v 详细）
ping / traceroute                      # 连通性 / 路由追踪
scp user@host:path .                   # 远程拷贝
ssh user@host                          # 远程登录
```

## 5. 文本处理三剑客（面试必背）

- **grep** 搜索：`grep -rn "pattern" .`（-r 递归 -n 行号）；`grep -v` 反选、`-i` 忽略大小写、`-E` 扩展正则；
- **sed** 流编辑：`sed -n '2,5p' file`（打印 2-5 行）、`sed 's/old/new/g'`（替换）、`sed -i` 就地修改；
- **awk** 列处理：`awk '{print $1, $2}'`（按空格/制表切列）、`awk -F, '{print $NF}'`（按逗号切、取最后一列）、`awk '$3 > 100'`（行过滤）、`awk '{s+=$1} END{print s}'`（求和）。

```bash
# 经典组合：管道 | 串起三剑客
ps -ef | grep python | grep -v grep | awk '{print $2}'   # 找 python 进程 PID
cat access.log | awk '{print $1}' | sort | uniq -c | sort -rn | head  # 访问 IP Top
df -h | awk 'NR>1 && $5+0 > 80 {print $1, $5}'          # 磁盘使用率>80% 的分区
```

## 6. 进程与信号

| 信号 | 默认动作 | 用途 |
| --- | --- | --- |
| SIGTERM(15) | 终止 | **优雅关闭**（程序可捕获做清理），`kill PID` 默认发这个 |
| SIGKILL(9) | 强杀 | 不可捕获、不可忽略，`kill -9` |
| SIGHUP(1) | 挂断 | 终端关闭；常被守护进程重载配置 |
| SIGINT(2) | 中断 | Ctrl+C |
| SIGSTOP/SIGCONT | 停/续 | Ctrl+Z / `fg` |

**孤儿进程 vs 僵尸进程**：

- **孤儿**：父进程先退出，子进程被 init(1) 收养——无害；
- **僵尸（Zombie）**：子进程已退出但父进程没 `wait` 回收，进程表项残留（`ps` 显示 `Z`）——父进程必须 `wait()` 清理，否则积累会耗尽 PID；`kill -9` 杀不死僵尸，要杀它的**父进程**。

## 7. 管道原理

- **管道（pipe）**把前一个进程的 stdout 接到后一个进程的 stdin：`A | B` = A 写、B 读，**两个进程并发**；
- 内核用**环形缓冲区**（大小有限，如 64KB）暂存；缓冲区满时写方阻塞，空时读方阻塞；
- 上次命令的退出码在 `$?`；管道中**多个命令并行**，`lastpipe`/`pipefail` 是 shell 选项。

## 8. shell 脚本基础

```bash
#!/bin/bash
set -e                     # 出错即退出（脚本健壮性）
name="world"
echo "Hello, $name"       # 变量双引号内展开
if [ -f /etc/passwd ]; then echo "exists"; fi
for i in {1..5}; do echo $i; done
while read line; do echo "$line"; done < file
case $1 in start) echo start;; stop) echo stop;; esac
func() { echo "args: $@"; }   # 函数，$@ 全部参数
```

**通配符**：`*` 任意串、`?` 单字符、`[abc]` 字符类；**引号**：`"..."` 保留变量展开、`'...'` 字面、反引号/`$(...)` 命令替换。

## 9. 面试 30 秒速答

- **Linux 是分层内核**：用户态经系统调用进内核态，安全隔离；
- **一切皆文件**：设备/进程信息以文件呈现（/dev、/proc）；
- **权限 rwx=421**：chmod 数字是每位和；umask 决定默认权限；SUID 以属主身份执行；
- **三剑客**：grep 搜、sed 改、awk 算（列处理）；`ps|grep|awk` 查 PID；
- **信号**：SIGTERM 优雅、SIGKILL 强杀；僵尸进程杀父进程；
- **管道**：A 的 stdout 进 B 的 stdin，环形缓冲、并发执行；
- **排查磁盘/内存**：`df -h`、`du -sh`、`free -h`、`top`。


## 核心概念

### (a) 🗂️ Linux：从「内核三层」到「命令与进程」的主线

Linux 的本体是一个**内核（kernel）**：它把硬件藏起来，向应用提供系统调用；而我们平时敲的命令、写的 shell，全都在内核之上的用户态里跑。把「内核分层 → 一切皆文件 → 权限模型 → 常用命令 → 进程信号」这条主线理清，八股就串起来了。

```mermaid
flowchart TB
    A["🧠 Linux 三层结构<br/>用户态 / 系统调用 / 内核态"]
    A --> B["📁 一切皆文件<br/>FHS 目录层次"]
    B --> C["🔐 权限模型<br/>rwx / 数字法 / SUID·SGID·Sticky"]
    C --> D["⌨️ 高频命令<br/>文件 / 进程 / 网络 / 文本"]
    D --> E["✂️ 文本三剑客<br/>grep / sed / awk"]
    A --> F["⚙️ 进程与信号<br/>fork / kill / 僵尸进程"]
    F --> G["📊 系统监控<br/>top / free / df"]
    E --> G
    G --> H["🎯 自测清单<br/>面试追问闭环"]
```

- **Linux 三层结构**：用户态的应用不能直接碰硬件，必须经过**系统调用**（read/write/fork/exec…）进入内核态；分层的目的就是安全隔离——普通进程碰不到别人的内存和硬件。
- **一切皆文件**：目录、设备（`/dev`）、内核信息（`/proc`）、管道、socket 都以文件形态存在，所以 FHS 目录层次（`/etc` 配置、`/var` 日志、`/tmp` 临时）必须熟。
- **权限模型**：r=4/w=2/x=1 的数字法（`chmod 755`）、符号法、`umask` 掩码，以及 SUID（借属主身份跑）、SGID（继承属组）、Sticky（/tmp 只能删自己的）三种特殊权限——面试高频。
- **高频命令**：文件（ls/cp/mv/rm/find）、进程（ps/top/kill/nohup）、网络（ss/netstat）、文本（cat/head/tail）按类记，看到命令能立刻说出用途和常见参数。
- **文本三剑客**：grep 行过滤、sed 流编辑、awk 列处理，配合管道 `|` 与 sort/uniq 做统计，是排查日志的看家本领。
- **进程与信号**：`kill -9`（SIGKILL 不可捕获）vs 默认 SIGTERM（可优雅退出）；子进程退出未被 wait 会变**僵尸进程**——面试常考怎么防。
- **系统监控**：top（动态进程）、free（内存）、df（磁盘）、du（目录大小），出现"排查问题"面试题时先报这几个命令就稳了。
- **自测清单**：每类命令能默写、能讲原理（权限怎么算、管道怎么跑、信号怎么处理），才算真正过关。

## 本节要解决的问题

| 要解决的问题 | 对应内容 |
|---|---|
| 程序凭什么能碰硬件而不乱来？ | 三层结构与系统调用 |
| 为什么说「一切皆文件」？ | FHS 目录层次与 /proc、/dev |
| `chmod 755` 里的数字怎么算出来的？ | rwx 数字法 + umask |
| SUID 为什么又常用又危险？ | 特殊权限 SUID / SGID / Sticky |
| 日志几万行怎么快速筛出问题？ | 文本三剑客 grep / sed / awk |
| 杀不掉的进程是怎么回事？ | 信号与僵尸进程 |
| 服务连不上先从哪查起？ | 网络排查 ss / netstat |
| 内存告警、磁盘写满怎么定位？ | 系统监控 top / free / df |

## 故事引入

凌晨两点，值班群炸了：线上接口突然全红。你登录服务器，第一件事是 `top` 看 CPU——一个进程把核吃满了；再 `ps -ef | grep java` 找到进程号，发现它是个老版本服务，日志还在 `/var/log/app` 下疯长。你 `tail -f` 盯日志，看到 `OutOfMemoryError` 不断刷屏；于是 `free -h` 确认内存被吃光，`df -h` 看磁盘也要满。你决定先把它优雅停掉——`kill <PID>` 发 SIGTERM 让它自己清理，几秒后没退，再 `kill -9` 强杀；然后顺手用 `grep`/`awk` 把最近的报错行统计出来定位到代码。

这一晚上你摸到的每个命令，背后全是 Linux 的设计：进程/内存/文件被内核管着，用户态只能通过系统调用请求；日志、内存、CPU 都「伪装」成文件躺在 `/proc`；权限位决定谁能动谁的文件；信号是内核和进程之间的「命令通道」。这篇就把这套底层逻辑一次讲透——**看懂原理，命令就不用背，全是自然推导**。

## 深入原理

### ① 三层结构：用户态、内核态与系统调用

**直觉**：内核是操作系统的「心脏」，直接管 CPU、内存、磁盘、网卡；普通程序（用户态）不能直接碰硬件——那样谁都能格式化磁盘就乱了。所以程序要做事（读文件、发网络包、开进程），只能通过**系统调用（syscall）**向内核「申请」，由内核核实权限后代为执行。

**要点**：典型系统调用有 `read`/`write`/`open`/`fork`/`exec`/`mmap`/`socket`。从用户态切到内核态要发生**特权级切换**（x86 的 ring 3 → ring 0），有固定开销，所以「系统调用密集」的程序（如大量小文件读）明显更慢。

**面试怎么问**：「一次 read 系统调用发生了什么？」——答：应用调 libc 封装 → 陷入内核 → 内核查文件描述符表 → 设备驱动取数据 → 拷回用户态缓冲区 → 返回。

![Linux 用户态、系统调用、内核态与硬件](images/linux_layers.svg)

> 📌 Linux 三层结构：用户程序只能通过系统调用请求内核代办，特权级切换保护硬件与数据

### ② FHS：/etc、/var、/proc 各自管什么

**直觉**：Linux 把整个系统组织成一棵目录树（FHS，文件系统层次标准），并且**一切皆文件**——设备是文件（`/dev`）、运行中的内核信息是文件（`/proc`）、socket 也能当文件用。记目录就是记「什么东西放在哪」的约定。

**要点**：`/etc` 放配置、`/var/log` 放日志、`/tmp` 放临时（重启清空，sticky 权限 1777）、`/home` 是用户家、`/proc` 是虚拟文件系统（`/proc/meminfo`、`/proc/cpuinfo`、每进程 `/proc/<pid>/`）。

**面试怎么问**：「/proc 是磁盘上的目录吗？」——不是，它是内核暴露的运行时视图，内容由内核实时生成，不占磁盘。

### ③ 权限数字法：r=4、w=2、x=1 与 umask

**直觉**：每个文件有三组权限（属主/属组/其他），每组三个位 rwx。把 rwx 看成二进制位就得到数字：r=4、w=2、x=1，`7=rwx`、`5=r-x`、`6=rw-`。`chmod 755` = 属主 7 + 属组 5 + 其他 5。

**要点**：`umask` 是「默认权限遮罩」：新建文件默认 `666 - umask`（文件无 x 位惯例），目录 `777 - umask`。`umask 022` → 文件 644、目录 755。注意是**按位取反再做与**（`base & ~umask`），不是简单减法。

**面试怎么问**：「umask 022 新文件权限多少？」——666 与 ~022（即 755）按位与 = 644。

![Linux rwx 权限、chmod 数字与 umask](images/linux_perm.svg)

> 📌 权限数字法 r=4/w=2/x=1 与 umask 默认权限计算：666 & ~022 = 644

### ④ 特殊权限：SUID / SGID / Sticky

**直觉**：普通用户改不了 `/etc/shadow`，但 `passwd` 命令却能改——靠的是 **SUID**：执行时进程临时获得**文件属主**的身份（root）。这是「借身份执行」的机制，也是提权漏洞的高发区。

**要点**：`rws` 中的 `s` 是 SUID（数字 4），`r-xr-s` 的 `s` 是 SGID（数字 2，目录上设置后新文件继承属组），`/tmp` 的 `t` 是 Sticky（数字 1，只有属主/root 能删自己的文件）。

**面试怎么问**：「为什么 /tmp 任何人都能写，但不能删别人的文件？」——Sticky 位保证目录里只能删自己的条目。

### ⑤ 文本三剑客：grep / sed / awk 的分工

**直觉**：排查日志的三大工具：**grep** 按行「找」（过滤行）、**sed** 按行「改」（流式编辑）、**awk** 按行「拆」（按列处理+统计）。配合管道 `|` 像流水线：`cat log | grep ERROR | awk '{print $2}' | sort | uniq -c`。

**要点**：grep 用正则（`grep -E` 扩展正则、`-v` 反选）；sed 常用 `s/旧/新/g` 替换、`-n '3,5p'` 打印区间；awk 默认按空白分列，`$1`/`$2` 取列，`BEGIN`/`END` 做前后处理。

**面试怎么问**：「统计日志里每个 IP 出现次数，一条命令？」——`awk '{print $1}' log | sort | uniq -c | sort -rn`。

![Linux 文本处理管道与标准流](images/linux_pipeline.svg)

> 📌 文本三剑客与管道：grep 过滤 → awk 拆列 → sort|uniq 统计，一行命令串成流水线

### ⑥ 进程与信号：SIGTERM vs SIGKILL、僵尸进程

**直觉**：信号是内核发给进程的「消息」。`kill` 不叫「杀死」，叫「发信号」。默认 `kill PID` 发 SIGTERM(15)——**可捕获**，程序能清理资源优雅退出；`kill -9` 发 SIGKILL(9)——**不可捕获不可忽略**，内核直接回收。

**要点**：子进程先于父进程退出、父进程没 `wait()` 收尸，子进程变**僵尸（zombie）**——已死但进程表条目还在，`ps` 里状态 Z。父进程崩了由 init 收养回收。

**面试怎么问**：「僵尸进程怎么产生的、怎么清？」——子进程退出未 wait；清法：kill 父进程让 init 接管，或父进程正确 wait/reap。

In [ ]:
# ---- matplotlib 中文字体（防图内中文乱码）----
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Songti SC', 'Arial Unicode MS', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
# -------------------------------------------
# 权限位计算模拟：rwx 数字法 + 符号法 + umask

BITS = {'r': 4, 'w': 2, 'x': 1}

def to_num(sym):
    """rwxr-xr-x -> 755（按三位一组编码为十进制）"""
    out = []
    for i in range(0, len(sym), 3):
        group = sym[i:i + 3]
        v = 0
        for ch in group:
            if ch in BITS:
                v += BITS[ch]
        out.append(str(v))
    return int(''.join(out))

def to_sym(num):
    """754 -> rwxr-xr--"""
    out = []
    for digit in str(num):
        d = int(digit)
        s = ''
        s += 'r' if d & 4 else '-'
        s += 'w' if d & 2 else '-'
        s += 'x' if d & 1 else '-'
        out.append(s)
    return ''.join(out)

print('rwxr-xr-- =', to_num('rwxr-xr--'))
print('755 ->', to_sym(755))
print('640 ->', to_sym(640))

# umask 计算默认权限：文件 666 - umask，目录 777 - umask
def default_perm(umask_val, is_dir=False):
    base = 0o777 if is_dir else 0o666
    return base & ~umask_val

u = 0o022
print(f'umask 022 -> 文件 {to_sym(default_perm(u))}，目录 {to_sym(default_perm(u, True))}')
assert to_num(to_sym(755)) == 755

In [ ]:
# 模拟 awk 列处理 + grep 行过滤 + sort|uniq 统计（理解管道语义）

log_lines = [
    '10.0.0.1 GET /api/a',
    '10.0.0.2 GET /api/b',
    '10.0.0.1 GET /api/a',
    '10.0.0.3 GET /api/c',
    '10.0.0.2 GET /api/b',
]

def pipe_awk_grep(lines, col, grep_pat=None):
    """模拟 grep 'pat' | awk '{print $col}'（$col 为 1 起始）"""
    res = []
    for ln in lines:
        if grep_pat and grep_pat not in ln:
            continue
        parts = ln.split()
        if len(parts) >= col:
            res.append(parts[col - 1])
    return res

# awk '{print $1}'：取每行第一列（IP）
ips = pipe_awk_grep(log_lines, 1)
print('awk $1:', ips)

# sort | uniq -c | sort -rn：计数排序
from collections import Counter
cnt = Counter(ips)
top = sorted(cnt.items(), key=lambda x: -x[1])
print('IP Top（sort|uniq -c|sort -rn）:')
for ip, n in top:
    print(f'  {n:>2}  {ip}')

# grep '/api/a' | awk '{print $2}'
paths = pipe_awk_grep(log_lines, 2, grep_pat='/api/a')
print('grep /api/a 的路径:', paths)
assert top[0] == ('10.0.0.1', 2)

In [ ]:
# 模拟进程信号：SIGTERM 优雅退出 vs SIGKILL 强杀 + 僵尸进程

class Signal:
    def __init__(self, num, name, catchable):
        self.num, self.name, self.catchable = num, name, catchable

SIGTERM = Signal(15, 'SIGTERM', True)
SIGKILL = Signal(9, 'SIGKILL', False)

def process(name, sig, has_handler=False):
    print(f'发送 {sig.name}({sig.num}) 给进程 {name}')
    if not sig.catchable:
        return f'{name}: 无法捕获 {sig.name}，立即被杀'
    if has_handler:
        return f'{name}: 捕获 {sig.name}，执行清理（保存数据/关连接）后退出'
    return f'{name}: 无处理函数，默认终止'

print(process('web-server', SIGTERM, has_handler=True))
print(process('web-server', SIGKILL))
print('<- 所以先 SIGTERM 优雅停机，超时再 SIGKILL（生产常用两步法）')

# 僵尸进程：子进程退出但父进程未 wait
class ZombieSim:
    def __init__(self, parent):
        self.parent = parent
        self.state = 'RUNNING'

    def exit_but_unreaped(self):
        self.state = 'ZOMBIE'   # 已退出，进程表项未回收
        print(f'子进程已退出但父进程({self.parent})没 wait -> 僵尸 Z')

    def reaped(self):
        self.state = 'REAPED'
        print('父进程 wait() 回收 -> 进程表项清除')

z = ZombieSim('app')
z.exit_but_unreaped()
print('ps 显示 Z 状态；kill -9 杀不掉（已死），要处理父进程')
z.reaped()

## 10. 自测清单（面试追问）

- [ ] 用户态与内核态的区别？为什么需要系统调用？举 3 个常见 syscall？
- [ ] 一切皆文件体现在哪？/proc 是什么？为什么要看 /proc/meminfo？
- [ ] rwx 数字怎么算？`chmod 754` 是什么？umask 022 下新建文件权限？
- [ ] SUID / SGID / Sticky 各是什么？/tmp 为什么是 1777？
- [ ] 手写：找监听 8000 端口的进程；找 python 进程 PID 并优雅杀掉。
- [ ] grep / sed / awk 各自擅长什么？写一个 awk 求和、sed 替换、grep 递归搜索。
- [ ] SIGTERM 与 SIGKILL 区别？为什么先 TERM 后 KILL？
- [ ] 孤儿进程 vs 僵尸进程？僵尸怎么产生、怎么清理？
- [ ] 管道 A | B 是怎么工作的？缓冲区满了怎么办？管道里命令并行吗？
- [ ] 软链接 vs 硬链接区别？（软链可跨文件系统、可指向目录、删源失效；硬链接同一 inode 计数）
- [ ] `du -sh` vs `df -h` 区别？进程被 nohup 后为什么还能跑？
- [ ] 排查「端口被占用」的命令序列？（ss -tlnp / lsof -i:port / netstat）
- [ ] 手写：统计 access.log 里访问量 Top5 的 IP；监控脚本每天检查磁盘 80% 告警。

> ✅ 全部能回答 + 能默写命令，本课通过。

## 面试考点（Q&A）

**Q1：一次 `read()` 系统调用的完整路径？**
A：应用 → libc 封装 → 陷入内核（ring0）→ 按 fd 找到文件 → 设备驱动读数据 → 拷到用户缓冲区 → 返回。💡 追问：为什么频繁小 read 慢？——每次都要特权切换与拷贝。

**Q2：`umask 022` 新建文件权限？**
A：`666 & ~022 = 644`；目录 `777 & ~022 = 755`。💡 追问：`umask 002` 呢？——文件 664、目录 775。

**Q3：SUID 的权限位长什么样、有什么风险？**
A：属主位 `rws`（如 `/usr/bin/passwd`），执行时以属主身份运行；风险：恶意二进制带 SUID 可提权。💡 追问：怎么找全系统 SUID 文件？——`find / -perm -4000`。

**Q4：僵尸进程和孤儿进程区别？**
A：僵尸=已死未被父进程回收（状态 Z）；孤儿=父进程先死被 init 收养，正常继续跑。💡 追问：大量僵尸怎么办？——修父进程的 wait 逻辑，或重启父进程。

**Q5：一条命令统计日志各 IP 次数？**
A：`awk '{print $1}' log | sort | uniq -c | sort -rn`。💡 追问：为什么先 sort 再 uniq？——uniq 只合并相邻重复行。

**Q6：`kill -9` 一定安全吗？**
A：不安全——SIGKILL 不可捕获，进程来不及落盘/清理锁，可能损坏数据；优先 SIGTERM。💡 追问：SIGTERM 之后程序没反应怎么办？——等几秒再升级 SIGKILL。

## 小结与下一步

Linux 不是「命令背诵大全」，而是一套**内核管资源、用户态走系统调用、一切皆文件、权限用位表达**的逻辑体系。把「三层结构 → 文件与权限 → 命令（找/改/拆）→ 进程信号」这条主线记住，遇到任何排查题都能从原理推导出命令。

**下一步**：下一篇 Shell 脚本把这些命令**串成自动化**——变量展开、管道、条件循环，让机器替你干活。